## NG-Notebook 2: Segment gaze classes with MobileSAM.

####
This notebook uses MobileSAM to segment objects in a video given a gaze estimate from Tobii eye-tracking glasses. This is the second notebook of the NeoGaze project, which aims to quantify physician visual attention during neonatal resuscitation.

We acknowledge the developers of Segment Anything Model (SAM) and MobileSAM for their work. Please cite their papers if you use this code.

Questions or issues? Please open an issue on GitHub.

#### 0. Install MobileSAM

In [7]:
!pip install git+https://github.com/ChaoningZhang/MobileSAM.git
!wget https://raw.githubusercontent.com/ChaoningZhang/MobileSAM/master/weights/mobile_sam.pt

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/ChaoningZhang/MobileSAM.git to c:\users\felipe parodi\appdata\local\temp\pip-req-build-wwsw34nu
  Resolved https://github.com/ChaoningZhang/MobileSAM.git to commit 01ea8d0f5590082f0c1ceb0a3e2272593f20154b
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for mobile-sam: filename=mobile_sam-1.0-py3-none-any.whl size=42698 sha256=a5b1bcaa99962117f7ddbe8b64f0194fff97daa0a15fd205f9666bf4264cf423
  Stored in directory: C:\Users\Felipe Parodi\AppData\Local\Temp\pip-ephem-wheel-cache-_kzp7bn5\wheels\b3\21\20\d16a72edc108d71267e9834f68e989ec70a47b761977cf74b3
Successfully built mobile-sam


  Running command git clone --filter=blob:none --quiet https://github.com/ChaoningZhang/MobileSAM.git 'C:\Users\Felipe Parodi\AppData\Local\Temp\pip-req-build-wwsw34nu'
'wget' is not recognized as an internal or external command,
operable program or batch file.


#### 1. Set up environment

In [12]:
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2
from PIL import Image

In [25]:
# Helper functions:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)
    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def get_greenest_pixel(path2image):
    """
    Returns the pixel with the highest green channel value.
    """
    img = Image.open(path2image)
    image_array = np.array(img)

    # Calculate green channel deviation
    green = image_array[:,:,1] - image_array.mean(axis=2)
    
    # Find the most green pixel in the image
    argmax2d_green = np.unravel_index(np.argmax(green), green.shape)

    # convert to array in the form of np.array([[x,y]]):
    argmax2d_green = np.array([[argmax2d_green[1], argmax2d_green[0]]])
    
    return argmax2d_green # returns tuple of (x,y) coordinates  

#### 2. Load testing image

In [17]:
# Load image
image_path = r'y:\neogaze\neogaze_deepethogram\DATA\Recording001 mf 2.10.22 from 00.00.00 to 00.02.18\frames\vid1_14_Vitals-Monitor.png'
image = cv2.imread(image_path)
input_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Display image
plt.figure(figsize=(10, 10))
plt.imshow(input_image)
plt.axis('off')
plt.show()

<Figure size 1000x1000 with 1 Axes>

In [67]:
%time
## ==== Get greenest point to use as input to MobileSAM ====

# Produce image embedding
predictor.set_image(input_image)

# Generate input point
input_point = get_greenest_pixel(image_path)
input_label = np.array([1])

print(f'Input point: {input_point}')
print(f'Input label: {input_label}')

CPU times: total: 0 ns
Wall time: 0 ns
Input point: [[1090  375]]
Input label: [1]


#### 3. Load MobileSAM

In [10]:
# Load MobileSAM
from mobile_sam import sam_model_registry, SamPredictor
sam_checkpoint = r'mobile_sam.pt'
model_type = "vit_t"

device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
sam.eval()

predictor = SamPredictor(sam)

CPU times: total: 0 ns
Wall time: 0 ns


#### 4. Run MobileSAM and visualize results

In [59]:
%time
# Generate segmentation mask with MobileSAM:
masks,scores,logits = predictor.predict(
    point_coords = input_point,
    point_labels = input_label,
    multimask_output=True, # keep true to select the best mask
)

print(f'Shape is {masks.shape}')

for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.figure(figsize=(10,10))
    plt.imshow(image)
    show_mask(mask, plt.gca())
    show_points(input_point, input_label, plt.gca())
    plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
    plt.axis('off')
    plt.show()

CPU times: total: 0 ns
Wall time: 0 ns
Shape is (3, 1080, 1920)


<Figure size 1000x1000 with 1 Axes>

<Figure size 1000x1000 with 1 Axes>

<Figure size 1000x1000 with 1 Axes>

In [66]:
# Keep the mask with has the highest area:
mask_areas = [np.sum(mask) for mask in masks]
best_mask = np.argmax(mask_areas)
print(f'Best mask: {best_mask}')
show_mask(masks[best_mask], plt.gca())

# Save mask as separate image:
cropped_image_array = image * masks[best_mask].reshape(1080, 1920, 1)
cropped_image = Image.fromarray(cropped_image_array)
cropped_image.save('cropped_image.png')
plt.figure(figsize=(10, 10))
plt.imshow(cropped_image)
plt.axis('off')
plt.show()

Best mask: 2


<Figure size 640x480 with 1 Axes>

<Figure size 1000x1000 with 1 Axes>

#### ...

In [ ]:

def segment_image(image_path):
    """
    Returns the segmented image.
    """
    # Load image
    image_path = r'y:\neogaze\neogaze_deepethogram\DATA\Recording001 mf 2.10.22 from 00.00.00 to 00.02.18\frames\vid1_14_Vitals-Monitor.png'
    image = cv2.imread(image_path)
    input_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Generate input point
    input_point = get_greenest_pixel(image_path)
    input_label = np.array([1])

    print(f'Input point: {input_point}')
    print(f'Input label: {input_label}')


    sam_checkpoint = r'mobile_sam.pt'
    model_type = "vit_t"

    device = "cuda" if torch.cuda.is_available() else "cpu"

    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
    sam.to(device=device)
    sam.eval()

    predictor = SamPredictor(sam)
    masks,scores,logits = predictor.predict(
        point_coords = input_point,
        point_labels = input_label,
        multimask_output=True, # keep true to select the best mask
    )
    cropped_image_array = image * masks[best_mask].reshape(1080, 1920, 1)
    cropped_image = Image.fromarray(cropped_image_array)

return cropped_image
# cropped_image.save('cropped_image.png')
# plt.figure(figsize=(10, 10))
# plt.imshow(cropped_image)
# plt.axis('off')
# plt.show()